# **Data Preprocessing**

## *Download*

Import needed modules.

In [ ]:
import requests
import os

from tqdm.notebook import tqdm

import huggingface_hub
import warnings

Define some parameters.

In [ ]:
# time interval for the dataset
YEAR: int = 2025
MONTH: int = 7

DAY_FROM: int = 1   # first day considered
DAY_TO: int = 7     # last day considered

# download
SOURCE: str = "andsanv/ais-tracks"
DOWNLOAD_PATH: str = "dataset/parquet/raw"

Download the data.

In [ ]:
# open huggingface apis
warnings.filterwarnings("ignore", message="The secret `HF_TOKEN` does not exist in your Colab secrets.")
warnings.filterwarnings("ignore", message="`local_dir_use_symlinks` parameter is deprecated")
api = huggingface_hub.HfApi()

# find parquet files
files = api.list_repo_files(SOURCE, repo_type="dataset")

# download files
for f in files:
    if f.endswith(".parquet"):
        huggingface_hub.hf_hub_download(
            repo_id=SOURCE,
            filename=f,
            repo_type="dataset",
            local_dir=DOWNLOAD_PATH,
            local_dir_use_symlinks=False
        )

## *Data Wrangling*

Import needed modules.

In [ ]:
import numpy as np
import pandas as pd
import polars as pl

Define parameters.

In [ ]:
# grid limits and cell parameters
TOP_LEFT: tuple[int, int] = (0.0, 60.0)
BOTTOM_RIGHT: tuple[int, int] = (20.0, 50.0)
CELL_DIMENSIONS: tuple[int, int] = (0.01, 0.01)   # dimensions of the single cell in the grid

Load dataframes from parquets.

In [ ]:
# create a list to store dataframes
raw_lazy_dfs: list = []

for day in tqdm(range(DAY_FROM, DAY_TO + 1), desc="Loading"):
    # lazily load dataset
    raw_lazy_df = pl.scan_parquet(f"{DOWNLOAD_PATH}/aisdk-{YEAR}-{MONTH:02d}-{day:02d}.parquet")    # load
    raw_lazy_dfs.append(raw_lazy_df)

## *Discretization*

Define some helpers for discretization.

In [ ]:
def discretize_segment(df: pl.DataFrame, cell_width: float, cell_height: float):
    """
    The method wrangles data by sampling only when a ship moves to a neighbor grid cell.
    It Calculates the 'dt' (time elapsed) since the last cell change.
    """

    out = (
        df
        .sort("timestamp")  # sort by time to ensure calculations are correct
        .with_columns([     # discretizes coordinates of the ships
            ((pl.col("x") - TOP_LEFT[0]) / cell_width).floor().cast(pl.Int32).alias("new_x"),
            ((pl.col("y") - BOTTOM_RIGHT[1]) / cell_height).floor().cast(pl.Int32).alias("new_y"),
        ])
        .with_columns([     # compare current sample's position with previous sample's position
            pl.col("new_x").shift(1).alias("prev_new_x"),
            pl.col("new_y").shift(1).alias("prev_new_y"),
        ])
        .filter(    # only keep rows if the sample is a new cell or it's the first point of the segment
            (pl.col("new_x") != pl.col("prev_new_x")) |
            (pl.col("new_y") != pl.col("prev_new_y")) |
            (pl.col("prev_new_x").is_null())
        )
        .with_columns([     # computes deltas of some attributes with respect to the previous sample
            # time
            (pl.col("timestamp") - pl.col("timestamp").shift(1))
            .dt.total_seconds()
            .fill_null(0.0) # first point has dt=0
            .cast(pl.Float32)
            .alias("dt"),

            # SOG
            (pl.col("SOG") - pl.col("SOG").shift(1))
            .fill_null(0.0)
            .cast(pl.Float32)
            .alias("dSOG"),

            # COG_sin and COG_cos
            (
                (pl.col("COG_sin") * pl.col("COG_cos").shift(1)) -
                (pl.col("COG_cos") * pl.col("COG_sin").shift(1))
            ).alias("sin_diff"),
            (
                (pl.col("COG_cos") * pl.col("COG_cos").shift(1)) +
                (pl.col("COG_sin") * pl.col("COG_sin").shift(1))
            ).alias("cos_diff")
        ])
        .with_columns([     # compute the difference angle in radians
            pl.arctan2(pl.col("sin_diff"), pl.col("cos_diff"))  # arctan2 returns a value in [-pi, pi]
            .fill_null(0.0)
            .cast(pl.Float32)
            .alias("dCOG")
        ])
        .drop(["x", "y", "prev_new_x", "prev_new_y", "sin_diff", "cos_diff"])   # remove helper columns
        .select([   # restore metadata
            pl.col("timestamp"),
            pl.col("MMSI"),
            pl.col("type"),
            pl.col("segment"),
            pl.col("new_x").alias("x"),     # change names of discretized coordinates to original coordinates' names
            pl.col("new_y").alias("y"),
            pl.col("SOG"),
            pl.col("COG_sin"),
            pl.col("COG_cos"),
            pl.col("dt"),
            pl.col("dSOG"),
            pl.col("dCOG"),
        ])
    )

    return out

Discretize the dataset by keeping only one sample per grid cell and embedding time through a new attribute.

In [ ]:
# define the output schema
output_schema = {
    "timestamp": pl.Datetime,
    "MMSI": pl.Int32,
    "type": pl.Categorical,
    "segment": pl.UInt32,
    "x": pl.Int32,
    "y": pl.Int32,
    "SOG": pl.Float32,
    "COG_sin": pl.Float32,
    "COG_cos": pl.Float32,
    "dt": pl.Float32,
    "dSOG": pl.Float32,
    "dCOG": pl.Float32,
}

# create target list and iterate through all days
lazy_dfs: list = []

for i in tqdm(range(len(raw_lazy_dfs)), desc="Discretization"):
    lazy_dfs.append(
        (
            raw_lazy_dfs[i]
            .group_by(["MMSI", "segment"])
            .map_groups(
                lambda group_df: discretize_segment(group_df, CELL_DIMENSIONS[0], CELL_DIMENSIONS[1]),
                schema=output_schema
            )
        )
    )

# **Models**

### *Prepare tensors*

Define needed modules.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

Define some parameters.

In [ ]:
# dataset dimension
DAYS: int = DAY_TO - DAY_FROM + 1   # compute total number of days in the dataset and divide 80-20
SPLIT_INDEX: int = int(DAYS * 0.9)

# tensors dimensions
WINDOW_SIZE: int = 20
BATCH_SIZE: int = 512

# training
LAMBDA_REGRESSION: float = 20.0         # used to scale regression loss and balance it with classification
EPOCHS: int = 10

Split into training and validation set.

In [ ]:
train_lazy_dfs: list = lazy_dfs[:SPLIT_INDEX]
val_lazy_dfs: list = lazy_dfs[SPLIT_INDEX:]

print(f"total = {DAYS} days\n - training: {len(train_lazy_dfs)} days\n - validation: {len(val_lazy_dfs)} days")

Define helpers.

In [ ]:
def compute_normalization_stats(dfs):
    """
    Computes maximum values of certain attributes, later used to normalize the data.
    """

    combined = pl.concat(dfs)   # single dataframe containing all samples

    # compute some statistics used later for normalization
    stats = combined.select([
        pl.col("x").max().alias("x_max"),       # maximum coordinates
        pl.col("y").max().alias("y_max"),
        pl.col("SOG").max().alias("SOG_max"),   # maximum speed
        pl.col("dSOG").abs().quantile(0.999).alias("dSOG_scale"),  # 99.9% quantile of acceleration to avoid edge cases (given by possible GPS glitches)
        (pl.col("dt").log1p().max()).alias("log_dt_max")            # delta time can be very high, therefore maximum of the log is considered
    ]).collect()

    return {
        "x_max": stats["x_max"][0],
        "y_max": stats["y_max"][0],
        "SOG_max": stats["SOG_max"][0],
        "dSOG_scale": stats["dSOG_scale"][0],
        "log_dt_max": stats["log_dt_max"][0]
    }



def create_dataset_tensors(dfs_list, stats, seq_len, train=True):
    """
    Divides the dataset into normalized tensors, used for training and validation of the model.
    """

    # prepare some structures
    X_MAX, Y_MAX, SOG_MAX, DSOG_SCALE, DT_LOG_MAX = stats["x_max"], stats["y_max"], stats["SOG_max"], stats["dSOG_scale"], stats["log_dt_max"]  # unpack statistics
    Xs = []; TYPEs = []; YCLAs = []; YREGs = []     # define lists used to store tensors
    type_map = {"Class A": 0, "Class B": 1}         # mapping from type to integer
    input_cols = ["x_norm", "y_norm", "sog_norm", "COG_sin", "COG_cos", "dt_norm", "dsog_norm", "dcog_norm"]    # attributes in input to the model
    reg_target_cols = ["sog_norm", "COG_sin", "COG_cos", "dt_norm", "dsog_norm", "dcog_norm"]                   # attributes target of the regression

    # process dataset
    for lf in tqdm(dfs_list, desc=f"Generating tensors ({"training" if train else "validation"} set)"):
        df = lf.collect()   # collect dataframes into memory

        # encode type and normalize
        df = df.with_columns([
            pl.col("type").cast(pl.String).replace_strict(type_map, default=0).cast(pl.Int32).alias("type_id"), # encode the type
            (pl.col("x") / X_MAX).cast(pl.Float32).alias("x_norm"),                 # normalize coordinates to interval [0, 1]
            (pl.col("y") / Y_MAX).cast(pl.Float32).alias("y_norm"),
            (pl.col("SOG") / SOG_MAX).cast(pl.Float32).alias("sog_norm"),           # normalize speed to interval [0, 1]
            (pl.col("dt").log1p() / DT_LOG_MAX).cast(pl.Float32).alias("dt_norm"),  # normalize time interval (log scale) to interval [0, 1]
            (pl.col("dSOG") / DSOG_SCALE).cast(pl.Float32).alias("dsog_norm"),      # normalize deltas to interval [-1, 1]
            (pl.col("dCOG") / np.pi).cast(pl.Float32).alias("dcog_norm"),
        ])

        # create classification targets (where the ship moves for the next step)
        df = df.with_columns([      # compute movement
            (pl.col("x").shift(-1).over(["MMSI", "segment"]) - pl.col("x")).fill_null(0).cast(pl.Int32).alias("next_dx"),
            (pl.col("y").shift(-1).over(["MMSI", "segment"]) - pl.col("y")).fill_null(0).cast(pl.Int32).alias("next_dy"),
        ])

        df = df.with_columns(       # translate the computed movement to the 3x3 grid (values between 0 and 9)
            ((pl.col("next_dy").clip(-1, 1) + 1) * 3 + (pl.col("next_dx").clip(-1, 1) + 1))     # formula is 3 * (dy + 1) + (dx + 1)
            .cast(pl.Int32).alias("target_cls")
        )

        # perform windowing on the data
        for _, group in df.group_by(["MMSI", "segment"]):   # group for every individual segment
            group = group.sort("timestamp")

            if len(group) < seq_len + 1: continue   # segment is useless if it does not fill the window (plus the target)
            if group["next_dx"].abs().max() > 2 or group["next_dy"].abs().max() > 2: continue   # handle case where data is glitched and huge jumps are performed by ships

            # convert to numpy
            features = group.select(input_cols).to_numpy()[:-1]         # don't care about last sample as it has no future target, therefore it's not used
            types = group.select("type_id").to_numpy().flatten()[:-1]

            # obtain targets
            targets_cla = group.select("target_cls").to_numpy().flatten()[seq_len:]    # we consider only samples after the first window
            targets_reg = group.select(reg_target_cols).to_numpy()[seq_len:]

            # create sliding windows
            windows = np.lib.stride_tricks.sliding_window_view(features, window_shape=(seq_len, len(input_cols)))
            windows = windows.squeeze(axis=1)

            # append to lists
            quantity = min(len(windows), len(targets_cla))  # since number of segments and targets might be different, only consider a number of segments which is the minimum between the two
            Xs.append(windows[:quantity]); TYPEs.append(types[:quantity])   # for type, only take the first type as it is constant per ship
            YCLAs.append(targets_cla[:quantity]); YREGs.append(targets_reg[:quantity])

    # concatenate the lists
    return (
        np.concatenate(Xs, axis=0),
        np.concatenate(TYPEs, axis=0),
        np.concatenate(YCLAs, axis=0),
        np.concatenate(YREGs, axis=0)
    )

Perform normalization on the train data.

In [ ]:
# compute statistics on the training data (no validation data, to avoid data leakage)
norm_stats = compute_normalization_stats(train_lazy_dfs)

print("Statistics")
for k, v in norm_stats.items():
    print(f"- {k}: {v}")

Prepare the dataset into tensors with their respective targets.

In [ ]:
# generate tensors
X_train, types_train, Ycla_train, Yreg_train = create_dataset_tensors(train_lazy_dfs, norm_stats, WINDOW_SIZE, train=True)
X_val, types_val, Ycla_val, Yreg_val = create_dataset_tensors(val_lazy_dfs, norm_stats, WINDOW_SIZE, train=False)

# print dimensions
print("\ngeneration complete.")
print(f" - training samples: {X_train.shape}")
print(f" - validation samples: {X_val.shape}")

Create a wrapper class for the dataset.

In [ ]:
class MaritimeDataset(Dataset):
    """
      A lightweight wrapper for pre-processed tensors. Assumes data are already normalized, windowed and targeted.
    """

    def __init__(self, x_num, x_type, y_cls, y_reg):  # convert numpy arrays to pytorch tensors
        self.x_num = torch.tensor(x_num, dtype=torch.float32)
        self.x_type = torch.tensor(x_type, dtype=torch.long)
        self.y_cls = torch.tensor(y_cls, dtype=torch.long)
        self.y_reg = torch.tensor(y_reg, dtype=torch.float32)

    def __len__(self):
        return len(self.x_num)

    def __getitem__(self, idx):
        return self.x_num[idx], self.x_type[idx], self.y_cls[idx], self.y_reg[idx]

Define datasets and loaders.

In [ ]:
# define the dataset
train_dataset = MaritimeDataset(X_train, types_train, Ycla_train, Yreg_train)
val_dataset = MaritimeDataset(X_val, types_val, Ycla_val, Yreg_val)

# prepare the loaders for model's training
train_loader = DataLoader(  # training dataset
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=True,
)

val_loader = DataLoader(    # validation dataset
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True,
)

### *Baseline*

Import needed modules.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

Implement the model.

In [ ]:
class RecentAverageBaseline(nn.Module):
    def __init__(self, device, n_last_steps, **kwargs):
        super().__init__()
        self.device = device
        self.n_last = n_last_steps  # indicates how many recent steps to average

        # grid moves (dx, dy) mapping to classes from 0 to 8
        self.vectors = torch.tensor([
            [-1, -1], [ 0, -1], [ 1, -1],
            [-1,  0], [ 0,  0], [ 1,  0],
            [-1,  1], [ 0,  1], [ 1,  1]
        ], dtype=torch.float32, device=device)

        # normalize movement vectors
        norms = self.vectors.norm(dim=1, keepdim=True); norms[4] = 1.0 # avoid division by zero for center
        self.unit_vectors = self.vectors / norms

    def forward(self, x_numeric, type_id=None):
        # slice taking all batches (:), last n steps (-n:), all features except x and y (2:)
        recent_window = x_numeric[:, -self.n_last:, 2:]

        # take mean over the time dimension (regression)
        y_reg_pred = recent_window.mean(dim=1)  # if the window is (B, 3, 6), this results in (B, 6)

        # compute and normalize predicted movement (classification)
        mean_sin = y_reg_pred[:, 1]; mean_cos = y_reg_pred[:, 2]    # direction computed based on sin and cos
        direction_vec = torch.stack([mean_sin, mean_cos], dim=1)    # dataset is such that 0 degrees is North
        dir_norm = direction_vec.norm(dim=1, keepdim=True)
        direction_vec = direction_vec / (dir_norm + 1e-8)

        # compute scores based on cosine similarity
        scores = torch.mm(direction_vec, self.unit_vectors.t())
        logits = scores * 10.0

        return logits, y_reg_pred

Evaluate the model.

In [ ]:
# pre-training setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   # use GPU if available

criterion_cls = nn.CrossEntropyLoss()   # define losses
criterion_reg = nn.MSELoss()

metrics = { # define metrics used to evaluate the model
    'loss': 0.0,
    'loss_cla': 0.0,
    'loss_reg': 0.0,
    'correct_preds': 0,
    'total_preds': 0
}


# evaluate the model
bl_model = RecentAverageBaseline(device=device, n_last_steps=20).to(device)
bl_model.eval()    # notify the model it's being evaluated

with torch.no_grad():
    for batch_x, batch_t, batch_y_cls, batch_y_reg in val_loader:
        # move to GPU
        batch_x = batch_x.to(device)
        batch_y_cls = batch_y_cls.to(device)
        batch_y_reg = batch_y_reg.to(device)

        # obtain predictions with forward pass
        logits, reg_preds = bl_model(batch_x)

        # compute losses
        loss_cla = criterion_cls(logits, batch_y_cls)
        loss_reg = criterion_reg(reg_preds, batch_y_reg)
        loss = loss_cla + (LAMBDA_REGRESSION * loss_reg)

        # update metrics
        batch_n = len(batch_x)
        metrics['loss'] += loss.item() * batch_n
        metrics['loss_cla'] += loss_cla.item() * batch_n
        metrics['loss_reg'] += loss_reg.item() * batch_n
        metrics['correct_preds'] += (torch.argmax(logits, dim=1) == batch_y_cls).sum().item()
        metrics['total_preds'] += batch_n


# visualize results
print(
        f"[evaluation] loss: {(metrics['loss'] / metrics['total_preds']):.3f}",
        f"(cla: {(metrics['loss_cla'] / metrics['total_preds']):.3f},",
        f"reg: {(LAMBDA_REGRESSION * metrics['loss_reg'] / metrics['total_preds']):.3f}),",
        f"acc: {(metrics['correct_preds'] / metrics['total_preds']):.2%}",
        sep=" "
    )

### *One Step RNN*

Import needed modules.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

Define some parameters.

In [ ]:
# model hyperparameters
HIDDEN_DIMENSION: int = 128
LAYERS_NUMBER: int = 1
EMBEDDING_DIMENSION: int = 2
DROPOUT: float = 0.1

# store and load
SAVE_MODEL: bool = False
LOAD_MODEL: bool = False

Implement the model.

In [ ]:
class MaritimeLSTM(nn.Module):
    def __init__(
        self,
        num_types: int,
        numeric_dim: int,
        hidden_dim: int,
        num_layers: int,
        embed_dim: int,
        num_classes: int = 9,
        dropout: float = 0.1,
    ):
        super().__init__()

        # embed the types
        self.type_embedding = nn.Embedding(num_types, embed_dim)

        # project the inputs
        input_total_dim = numeric_dim + embed_dim
        self.input_proj = nn.Sequential(    # this is a first layer to help LSTM to digest mixed inputs
            nn.Linear(input_total_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim)
        )

        # LTSM architecture
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        # heads of the RNN
        self.fc_cls = nn.Linear(hidden_dim, num_classes)    # head of the classification part (move on the grid)
        self.fc_sog = nn.Linear(hidden_dim, 1)              # heads of the regression part (assume target order: SOG_norm, COG_sin, COG_cos, dt_norm, dSOG_norm, dCOG_norm)
        self.fc_trig = nn.Linear(hidden_dim, 2)
        self.fc_dt = nn.Linear(hidden_dim, 1)
        self.fc_deltas = nn.Linear(hidden_dim, 2)


    def forward(self, x_numeric: torch.Tensor, type_id: torch.Tensor):
        batch_size, seq_len, _ = x_numeric.shape

        # embedding for the type
        type_emb = self.type_embedding(type_id)
        type_emb_expanded = type_emb.unsqueeze(1).expand(batch_size, seq_len, -1)

        # concatenate the inputs and project them
        x = torch.cat([x_numeric, type_emb_expanded], dim=-1)
        x = self.input_proj(x)

        # compute the output with the LSTM
        output, (hn, cn) = self.lstm(x)
        last_hidden = output[:, -1, :]      # we are only interested in the output of the LTSM at the last position

        # apply activation functions to enforce physics
        logits_cls = self.fc_cls(last_hidden)                       # classification (grid movement)
        predicted_sog = torch.sigmoid(self.fc_sog(last_hidden))     # SOG, sigmoid forces the output to be in [0, 1]
        predicted_trig = torch.tanh(self.fc_trig(last_hidden))      # COG_sin and COG_cos, tanh forces the output to be in [-1, 1]
        predicted_dt = F.softplus(self.fc_dt(last_hidden))          # dt, sotfplus (smooth relu) forces the output to be [0, inf]
        predicted_deltas = self.fc_deltas(last_hidden)              # other deltas, no constraints

        # return
        return logits_cls, torch.cat([predicted_sog, predicted_trig, predicted_dt, predicted_deltas], dim=1)    # returns classification results followed by regression results


    def predict_proba(self, x_numeric, x_type):
        self.eval()

        with torch.no_grad():
            logits_cls, _ = self.forward(x_numeric, x_type)
            return torch.softmax(logits_cls, dim=-1)

Define and load the model.

In [ ]:
# load gpu if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# define the model
s2o_model = MaritimeLSTM(
    num_types=2,
    numeric_dim=8,
    hidden_dim=HIDDEN_DIMENSION,
    num_layers=LAYERS_NUMBER,
    embed_dim=EMBEDDING_DIMENSION,
    num_classes=9,
    dropout=DROPOUT,
)

# load model parameters if boolean is set to True
if LOAD_MODEL:
    state_dict = torch.load("model.pth", map_location=device)  # load from file
    s2o_model.load_state_dict(state_dict)   # load into model

    if device.type == "cuda":   # compile the model for better performance, in case gpu is available
        s2o_model = torch.compile(s2o_model, mode="reduce-overhead")
        s2o_model = s2o_model.to(device)

    print("[INFO] \"LOAD_MODEL\" set to 'True', model overwritten with stored version.") # notify user

Prepare parameters for training.

In [ ]:
# apply settings for better efficiency for gpus (and RNNs with LSTM cells)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


# silence pytorch warnings
os.environ["TORCHINDUCTOR_DISABLE_WARNINGS"] = "1"; os.environ["PYTORCH_TRITON_DISABLE_WARNINGS"] = "1"
os.environ["TORCHINDUCTOR_LOG"] = "0"; os.environ["PYTORCH_JIT_LOG_LEVEL"] = "0"


# training
criterion_cls = nn.CrossEntropyLoss()   # define loss functions
criterion_reg = nn.MSELoss()

optimizer = torch.optim.AdamW(s2o_model.parameters(), lr=1e-3, fused=True)  # define weighted Adam as optimizer, fused to improve training speed
scaler = torch.amp.GradScaler('cuda')   # include gradient scaler for numerical stability


# other
if device.type == "cuda":   # compile the model for better performance, in case gpu is available
    s2o_model = torch.compile(s2o_model, mode="reduce-overhead")
    s2o_model = s2o_model.to(device)

Train the model.

In [ ]:
# preliminary ops
if SAVE_MODEL: print("[INFO] \"SAVE_MODEL\" set to 'True', saved model will be overwritten at the end of the run.")

epoch_metrics: dict = {
    'train_correct_preds': 0,  # int
    'train_total_preds': 0,    # int
    'train_loss': 0.0,         # float
    'train_loss_cla': 0.0,     # float
    'train_loss_reg': 0.0,     # float
    'val_correct_preds': 0,    # int
    'val_total_preds': 0,      # int
    'val_loss': 0.0            # float
}



# go through each epoch
for epoch in range(EPOCHS):
    # setup
    s2o_model.train()   # notify the model that it's being trained (fundamental for dropout)

    epoch_metrics['train_correct_preds'] = epoch_metrics['train_total_preds'] = 0   # zero the metrics
    epoch_metrics['train_loss'] = epoch_metrics['train_loss_cla'] = epoch_metrics['train_loss_reg'] = 0.0


    # train the model
    for x_num, x_type, y_cls, y_reg in (pbar := tqdm(train_loader, desc=f"Epoch {(epoch + 1):2d}")):
        # send data to GPU
        x_num = x_num.to(device, non_blocking=True)
        x_type = x_type.to(device, non_blocking=True)
        y_cls = y_cls.to(device, non_blocking=True)
        y_reg = y_reg.to(device, non_blocking=True)

        optimizer.zero_grad()           # reset (zero) the gradients
        s2o_model.lstm.flatten_parameters()

        # predict and compute losses
        with torch.amp.autocast('cuda'):
            logits, reg_preds = s2o_model(x_num, x_type)                # predict

            loss_cla = criterion_cls(logits, y_cls)                 # compute partial losses
            loss_reg = criterion_reg(reg_preds, y_reg)
            loss_tot = loss_cla + LAMBDA_REGRESSION * loss_reg      # compute total loss

        # backpropagate
        scaler.scale(loss_tot).backward()
        torch.nn.utils.clip_grad_norm_(s2o_model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        # update metrics
        batch_size = x_num.size(0)      # required in case total samples are not multiple of BATCH_SIZE
        epoch_metrics['train_loss'] += loss_tot.item() * batch_size       # losses
        epoch_metrics['train_loss_cla'] += loss_cla.item() * batch_size
        epoch_metrics['train_loss_reg'] += loss_reg.item() * LAMBDA_REGRESSION * batch_size
        epoch_metrics['train_correct_preds'] += (torch.argmax(logits, dim=1) == y_cls).sum().item()
        epoch_metrics['train_total_preds'] += batch_size


    # validate the model
    s2o_model.eval()  # notify the model that it's being evaluated (fundamental for dropout)

    epoch_metrics['val_loss'] = 0.0     # zero the metrics
    epoch_metrics['val_correct_preds'] = epoch_metrics['val_total_preds'] = 0

    with torch.no_grad():   # disable gradient computation for inference
        for x_num, x_type, y_cls, y_reg in val_loader:
            # send data to GPU
            x_num = x_num.to(device, non_blocking=True)
            x_type = x_type.to(device, non_blocking=True)
            y_cls = y_cls.to(device, non_blocking=True)
            y_reg = y_reg.to(device, non_blocking=True)

            with torch.amp.autocast('cuda'):
                logits, reg_preds = s2o_model(x_num, x_type)                # predict

                loss_cla = criterion_cls(logits, y_cls)                 # compute partial losses
                loss_reg = criterion_reg(reg_preds, y_reg)
                loss_tot = loss_cla + LAMBDA_REGRESSION * loss_reg      # compute total loss

            # update metrics
            batch_size = x_num.size(0)      # required in case total samples are not multiple of BATCH_SIZE
            epoch_metrics['val_loss'] += loss_tot.item() * batch_size   # losses
            epoch_metrics['val_correct_preds'] += (torch.argmax(logits, dim=1) == y_cls).sum().item()
            epoch_metrics['val_total_preds'] += batch_size


    # print epoch results
    print(
        f"[train] loss: {(epoch_metrics['train_loss'] / epoch_metrics['train_total_preds']):.3f}",
        f"(cla: {(epoch_metrics['train_loss_cla'] / epoch_metrics['train_total_preds']):.3f},",
        f"reg: {(epoch_metrics['train_loss_reg'] / epoch_metrics['train_total_preds']):.3f}),",
        f"acc: {(epoch_metrics['train_correct_preds'] / epoch_metrics['train_total_preds']):.2%}",
        f"| [validation] loss: {(epoch_metrics['val_loss'] / epoch_metrics['val_total_preds']):.3f}, acc: {(epoch_metrics['val_correct_preds'] / epoch_metrics['val_total_preds']):.2%}",
        sep=" "
    )



# save model if requested
if SAVE_MODEL:
    torch.save(s2o_model._orig_mod.state_dict(), "model.pth")

# **Visualization**

Import needed modules.

In [ ]:
!pip install contextily -q

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import contextily as cx

Define global parameters.

In [ ]:
RELATIVE_OFFSETS = torch.tensor([
    [-1, -1], [ 0, -1], [ 1, -1],
    [-1,  0], [ 0,  0], [ 1,  0],
    [-1,  1], [ 0,  1], [ 1,  1],
], dtype=torch.float32)

RANDOM_STATE: int = 16

### *Accuracy*

Define some parameters.

In [ ]:
SAMPLES_NUMBER: int = 65535  # amount of samples on which to test accuracy

Define helper.

In [ ]:
def evaluate_multistep_metrics(model, X_val, Y_cls_val, norm_stats, n_steps, n_samples, temperature=1e-5, batch_size=100, device="cuda", model_name = None):
    """
    Computes mean and standard deviation for accuracy over time.
    """

    model.eval()    # notify the model that it's being evaluated (fundamental for dropout)

    # Move constants
    moves_map = RELATIVE_OFFSETS.to(device)
    x_step_norm = 1.0 / norm_stats["x_max"]
    y_step_norm = 1.0 / norm_stats["y_max"]

    # store accuracy and distance (from true label) for every step of the window
    step_metrics = {step: {'accuracy': [], 'distance': []} for step in range(n_steps)}  # every position in the array is a different batch

    # sample random indices from the whole dataset length
    max_idx = len(X_val) - n_steps - 1  # ensures there are enough future data to evaluate the prediction
    np.random.seed(RANDOM_STATE)
    indices = np.random.choice(max_idx, n_samples, replace=False)   # sample randomly


    # simulate in batches
    for i in tqdm(range(0, n_samples, batch_size), desc="Simulating"):
        # prepare data
        batch_indices = indices[i : i + batch_size]     # extract batch indices
        current_batch_size = len(batch_indices)
        seeds = torch.tensor(X_val[batch_indices], dtype=torch.float32, device=device)  # import to pytorch
        batch_types = torch.tensor(types_val[batch_indices], dtype=torch.long, device=device)

        # compute ground truth and validate it
        ground_truth_cls = []
        valid_mask = torch.ones((current_batch_size, n_steps), dtype=torch.bool, device=device)     # prepare a mask to invalidate certain samples

        for step in range(n_steps):     # consider each step in window size
            gt_step = Y_cls_val[batch_indices + step]  # get samples at 'step' positions later position
            ground_truth_cls.append(gt_step)

            # segment continuity check
            prev_win_end = X_val[batch_indices + step, -1, :2]
            curr_win_prev = X_val[batch_indices + step + 1, -2, :2]
            dist = np.abs(prev_win_end - curr_win_prev).sum(axis=1)
            is_broken = dist > 1e-5

            if np.any(is_broken):
                broken_indices = np.where(is_broken)[0]
                valid_mask[broken_indices, step:] = False   # set to invalid segments which don't respect continuity


        # preliminary ops for prediction and simulation
        ground_truth_cls = torch.tensor(np.stack(ground_truth_cls, axis=1), device=device)

        pos_pred = torch.zeros((current_batch_size, 2), device=device)
        pos_true = torch.zeros((current_batch_size, 2), device=device)

        current_seq = seeds.clone()

        # compute predictions for each step
        for t in range(n_steps):
            # predict
            with torch.no_grad():
                logits, reg_preds = model(current_seq, batch_types)

                if temperature < 1e-4:  # just predict deterministically if temperature is too low
                    preds = torch.argmax(logits, dim=1)
                else:
                    probs = F.softmax(logits / temperature, dim=-1)
                    preds = torch.multinomial(probs, num_samples=1).squeeze(1)

            # update change of position throughout window of both predicted and true
            pos_pred += moves_map[preds]
            pos_true += moves_map[ground_truth_cls[:, t]]

            # compute and update metrics
            current_dist = torch.norm(pos_pred - pos_true, dim=1)   # distance
            is_correct = (preds == ground_truth_cls[:, t]).float()  # accuracy

            mask = valid_mask[:, t]     # only consider valid samples
            valid_accs = is_correct[mask].cpu().numpy()
            valid_dists = current_dist[mask].cpu().numpy()

            step_metrics[t]['accuracy'].append(valid_accs)  # append results to results list
            step_metrics[t]['distance'].append(valid_dists)

            # create a vector that serves as next state
            next_step_vec = torch.zeros((current_batch_size, 8), device=device)     # initialize all zeros
            prev_x = current_seq[:, -1, 0]
            prev_y = current_seq[:, -1, 1]
            next_step_vec[:, 0] = prev_x + (moves_map[preds][:, 0] * x_step_norm)    # compute new x and y based on previous prediction
            next_step_vec[:, 1] = prev_y + (moves_map[preds][:, 1] * y_step_norm)
            next_step_vec[:, 2:] = reg_preds    # include predicted regression values

            current_seq = torch.cat([current_seq[:, 1:, :], next_step_vec.unsqueeze(1)], dim=1)


    # report the results
    if model_name is not None:
        print(f"Simulation results for {model_name} model:")
    print(f"\n{'step':<5} | {'accuracy':<25} | {'distance':<25}")
    print("-" * 65)

    final_acc_mean = []
    final_dist_mean = []

    for t in range(n_steps):    # for each step
        # concatenate all batches
        accs = np.concatenate(step_metrics[t]['accuracy'])
        dists = np.concatenate(step_metrics[t]['distance'])

        if len(accs) > 0:
            acc_mean, acc_std = np.mean(accs), np.std(accs)
            dist_mean, dist_std = np.mean(dists), np.std(dists)

            final_acc_mean.append(acc_mean)
            final_dist_mean.append(dist_mean)

            print(f"{(t + 1):<5} | {(f"{acc_mean:.2%} ± {acc_std:.2f}"):<25} | {(f"{dist_mean:.2f} ± {dist_std:.2f}"):<25}")
        else:
            print(f"{(t + 1):<5} | n/a")

    print("-" * 65)
    print(f"overall average accuracy: {np.mean(final_acc_mean):.2%}")

    return final_acc_mean, final_dist_mean

Compute accuracies along the whole window.

In [ ]:
# baseline
evaluate_multistep_metrics(
    bl_model, X_val, Ycla_val, norm_stats,
    n_steps=2 * WINDOW_SIZE,    # consider 40 steps for completeness
    n_samples=SAMPLES_NUMBER,
    temperature=0,
    model_name="Baseline"
)

# deterministic seq2one
evaluate_multistep_metrics(
    s2o_model, X_val, Ycla_val, norm_stats,
    n_steps=2 * WINDOW_SIZE,
    n_samples=SAMPLES_NUMBER,
    temperature=1e-5,   # basically computing argmax at each step
    model_name="Deterministic SeqToOne"
)

# probabilistic seq2seq
evaluate_multistep_metrics(
    s2o_model, X_val, Ycla_val, norm_stats,
    n_steps=2 * WINDOW_SIZE,
    n_samples=SAMPLES_NUMBER,
    temperature=1,
    model_name="Probabilistic SeqToOne"
);

### *Simulation*

Define parameters.

In [ ]:
PLOT_ROWS: int = 2  # number of plotted elements, per column
PLOT_COLS: int = 2  # number of plotted elements, per row

STEPS_NUMBER: int = 40  # number of predicted next steps for each seed
RUNS_NUMBER: int = 75   # number of runs per seed

MARGIN: float = 0.05

Define helpers.

In [ ]:
def grid_to_lonlat(x_grid: int, y_grid: int, top_left: tuple, bottom_right: tuple, cell_dims: tuple) -> tuple[float, float]:
    """
    Converts grid indices to longitude and latitude.
    """

    longitude: float = top_left[0] + x_grid * cell_dims[0]        # compute longitude
    latitude: float = bottom_right[1] + y_grid * cell_dims[1]     # compute latitude

    return longitude, latitude



def get_true_trajectory(Xs, types, idx, n_steps, norm_stats):
    """
    Given a seed, returns the following 'n_steps' steps of the true trajectory.
    """

    # prepare structures
    true_path: list = []
    if idx + n_steps > len(Xs) - 1:     # sample is at the end of the dataset
        return None

    previous_window = Xs[idx]       # explicitly store the window at the given index

    X_MAX = norm_stats["x_max"]     # obtain normalization factors
    Y_MAX = norm_stats["y_max"]

    # obtain trajectory for all next samples
    for t in range(1, n_steps + 1):
        target_idx: int = idx + t           # update index
        current_window = Xs[target_idx]     # obtain the window starting at step t

        last_known = previous_window[-1, :2]    # take last position in previous window
        overlap_point = current_window[-2, :2]  # take second to last position in current window

        if not np.allclose(last_known, overlap_point, atol=1e-5):   # check that the two positions match
            # if the windows don't overlap, it means the samples are from two different ships
            return None

        grid_x = current_window[-1, :2][0] * X_MAX  # de-normalize
        grid_y = current_window[-1, :2][1] * Y_MAX
        true_path.append([grid_x, grid_y])    # append last point of new window to true trajectory

        previous_window = current_window   # update previous window

    # return true trajectory for next 'n_steps'
    return np.array(true_path)



def monte_carlo(model, Xs, types, idx, n_steps, n_samples, window_size, device, norm_stats, temperature = 1):
    """
    Returns a Monte Carlo simulation containing a prediction for the next 'n_steps' steps.
    The simulation considers a given seed, found at position 'idx' in the dataset 'X'.
    """

    # obtain the seed at position idx in the dataset
    seed_norm = Xs[idx]          # tensor related to one window (shape: (window_size, 8)), at position 'idx' in the dataset
    vessel_type = types[idx]     # obtain vessel type

    # de-normalize values
    IDX_X, IDX_Y, IDX_SOG, IDX_SIN, IDX_COS = 0, 1, 2, 3, 4     # define array indices for readability
    IDX_DT, IDX_DSOG, IDX_DANGLE = 5, 6, 7  #
    IDX_DX, IDX_DY = 8, 9   # made up features (not in the input) but useful for later

    X_MAX = norm_stats["x_max"]; Y_MAX = norm_stats["y_max"]     # unpack normalization statistics
    SOG_MAX = norm_stats["SOG_max"]; DSOG_SCALE = norm_stats["dSOG_scale"]; LOG_DT_MAX = norm_stats["log_dt_max"]

    seed_real = np.zeros((window_size, 10), dtype=np.float32)   # create a new array to hold un-normalized data
    seed_real[:, IDX_X] = seed_norm[:, IDX_X] * X_MAX       # de-normalize data
    seed_real[:, IDX_Y] = seed_norm[:, IDX_Y] * Y_MAX
    seed_real[:, IDX_SOG] = seed_norm[:, IDX_SOG] * SOG_MAX
    seed_real[:, IDX_SIN] = seed_norm[:, IDX_SIN]
    seed_real[:, IDX_COS] = seed_norm[:, IDX_COS]
    seed_real[:, IDX_DT]  = np.expm1(seed_norm[:, IDX_DT] * LOG_DT_MAX)
    seed_real[:, IDX_DSOG] = seed_norm[:, IDX_DSOG] * DSOG_SCALE
    seed_real[:, IDX_DANGLE] = seed_norm[:, IDX_DANGLE] * np.pi
    seed_real[:, IDX_DX] = np.diff(seed_real[:, IDX_X], prepend=seed_real[0, IDX_X])    # fill the helper columns dx, dy for the seed history
    seed_real[:, IDX_DY] = np.diff(seed_real[:, IDX_Y], prepend=seed_real[0, IDX_Y])


    # simulate following Monte Carlo approach
    rollouts: list = [] # define a list to store all paths
    model.eval()    # set model to evaluation mode (fundamental, turns off dropout)

    for s in range(n_samples):  # run 'n_samples' independent simulations
        sim_state = seed_real.copy()    # take a copy of the seed history to not interfere with other simulations

        for t in range(n_steps):    # step-by-step prediction loop
            # prepare the input for the model
            current_window = sim_state[-window_size:]
            norm_input = np.zeros((window_size, 8), dtype=np.float32)   # placeholder for the model input

            norm_input[:, IDX_X] = current_window[:, IDX_X] / X_MAX     # re-normalize every feature
            norm_input[:, IDX_Y] = current_window[:, IDX_Y] / Y_MAX
            norm_input[:, IDX_SOG] = current_window[:, IDX_SOG] / SOG_MAX
            norm_input[:, IDX_SIN] = current_window[:, IDX_SIN]
            norm_input[:, IDX_COS] = current_window[:, IDX_COS]
            norm_input[:, IDX_DT] = np.log1p(current_window[:, IDX_DT]) / LOG_DT_MAX
            norm_input[:, IDX_DSOG] = current_window[:, IDX_DSOG] / DSOG_SCALE
            norm_input[:, IDX_DANGLE] = current_window[:, IDX_DANGLE] / np.pi

            # predict next step
            x_tensor = torch.from_numpy(norm_input).unsqueeze(0).float().to(device)     # convert to pytorch tensor and move to gpu
            type_tensor = torch.tensor([vessel_type], dtype=torch.long, device=device)  # convert vessel type to tensor

            with torch.no_grad():   # turn off gradients as the model is just predicting
                logits_cls, y_reg_pred = model(x_tensor, type_tensor)       # infere
                scaled_logits = logits_cls / temperature                    # apply temperature scaling
                probs = F.softmax(scaled_logits, dim=-1)[0].cpu().numpy()   # apply softmax

                cls_out = np.random.choice(len(probs), p=probs) # choose randomly among the probabilites
                reg_out = y_reg_pred[0].cpu().numpy()           # obtain regression output

            # build next sample based on the prediction
            prev_vec = current_window[-1]               # explicitly store last position of the window
            next_vec = np.zeros(10, dtype=np.float32)   # create a new vector to store next position

            next_vec[IDX_X] = int(prev_vec[IDX_X]) + RELATIVE_OFFSETS[cls_out][0]   # update position (X, Y) for next vector based on prediction 'out'
            next_vec[IDX_Y] = int(prev_vec[IDX_Y]) + RELATIVE_OFFSETS[cls_out][1]
            next_vec[IDX_DX] = RELATIVE_OFFSETS[cls_out][0]                         # store offset explicitly
            next_vec[IDX_DY] = RELATIVE_OFFSETS[cls_out][1]

            next_vec[IDX_SOG] = reg_out[0] * SOG_MAX                            # update computed regression attributes
            next_vec[IDX_SIN] = reg_out[1]
            next_vec[IDX_COS] = reg_out[2]
            next_vec[IDX_DT]  = max(0.0, np.expm1(reg_out[3] * LOG_DT_MAX))
            next_vec[IDX_DSOG] = reg_out[4] * DSOG_SCALE
            next_vec[IDX_DANGLE] = reg_out[5] * np.pi

            # add the predicted next sample to the simulation history
            sim_state = np.vstack([sim_state, next_vec])


        # store the result
        rollouts.append(np.stack([sim_state[window_size:, IDX_X], sim_state[window_size:, IDX_Y]], axis=1))  # skip the seed, only store predictions

    return np.stack(rollouts, axis=0)   # 'rollouts' list of arrays is converted into one 3D array (samples, steps, 2)



def plot_simulation(
        model, Xs, types, idx, true_path, sim_paths, n_steps, n_runs, window_size, norm_stats, top_left, bottom_right, cell_dims, ax
):
    """
    Plots a simulation, including seed, true path and predicted path(s).
    """

    # retrieve seed and de-normalize it
    seed_norm = Xs[idx]
    seed_xs = np.round(seed_norm[:, 0] * norm_stats['x_max'])
    seed_ys = np.round(seed_norm[:, 1] * norm_stats['y_max'])

    # transform seed, true path and simulation results into continuous (real) coordinates
    seed_xs, seed_ys = grid_to_lonlat(np.round(seed_xs), np.round(seed_ys), top_left, bottom_right, cell_dims)
    true_xs, true_ys = grid_to_lonlat(np.round(true_path[:, 0]), np.round(true_path[:, 1]), top_left, bottom_right, cell_dims)

    sim_paths = np.round(sim_paths)
    sim_xs, sim_ys = grid_to_lonlat(sim_paths[:, :, 0].flatten(), sim_paths[:, :, 1].flatten(), top_left, bottom_right, cell_dims)

    # find boundaries for the map
    all_xs = np.concatenate([seed_xs, true_xs, sim_xs])     # combine all points
    all_ys = np.concatenate([seed_ys, true_ys, sim_ys])

    min_lon, max_lon = all_xs.min() - MARGIN, all_xs.max() + MARGIN     # find mins and maxs
    min_lat, max_lat = all_ys.min() - MARGIN, all_ys.max() + MARGIN
    size = max(max_lon - min_lon, max_lat - min_lat)    # compute the biggest between width and height

    mid_point_lon = (max_lon + min_lon) / 2.0       # compute midpoints
    mid_point_lat = (max_lat + min_lat) / 2.0

    # create bins, offset of half a cell for numerical stability and clarity
    lon_bins = np.arange(min_lon, max_lon, cell_dims[0]) - (CELL_DIMENSIONS[0] / 2.0)
    lat_bins = np.arange(min_lat, max_lat, cell_dims[1]) - (CELL_DIMENSIONS[1] / 2.0)


    # plot the heatmap of predictions
    ax.hist2d(
        sim_xs, sim_ys,
        bins=[lon_bins, lat_bins],
        cmap='turbo',
        cmin=1,      # color only cells with a count of at least 1
        alpha=0.5,   # include transparency to make map visible
        zorder=2
    )

    # plot seed with a uniform black line
    ax.plot(seed_xs, seed_ys, color='black', linewidth=2.5, label='seed', zorder=5)     # plot whole seed
    ax.scatter(seed_xs[-1], seed_ys[-1], color='black', s=60, zorder=5)     # scatter a point on the most recent sample

    # plot true trajectory with a dashed black line
    traj_xs = np.concatenate(([seed_xs[-1]], true_xs))   # connect last seed point to first truth point for continuity
    traj_ys = np.concatenate(([seed_ys[-1]], true_ys))
    ax.plot(traj_xs, traj_ys, color='black', linewidth=2.5, linestyle='--', label='truth', zorder=6)

    # set square limits based on trajectory
    ax.set_xlim(mid_point_lon - size / 2, mid_point_lon + size / 2)
    ax.set_ylim(mid_point_lat - size / 2, mid_point_lat + size / 2)
    ax.set_aspect('equal')  # to ensure map is not warped

    # load world map
    try:
        cx.add_basemap(ax, crs="EPSG:4326", source=cx.providers.CartoDB.Positron)
    except Exception as e:
        print(f"[ERROR] Could not load map.")


    # plot settings
    ax.legend(loc='upper right', fontsize=20)
    # ax.set_title(f"Sample Index: {idx}", fontsize=10)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

Compute predictions.

In [ ]:
# setup figure
fig, axes = plt.subplots(PLOT_ROWS, PLOT_COLS, figsize=(15, 15))
axes = axes.flatten()

# loop
valid_samples: int = 0      # counts valid samples plotted

while valid_samples < PLOT_ROWS * PLOT_COLS:    # only stop when as many valid predictions as axes have been found
    index = np.random.choice(len(X_val), replace=False)     # pick randomly one window from validation set

    # get paths
    true_path = get_true_trajectory(X_val, types_val, index, STEPS_NUMBER, norm_stats)      # true path
    if true_path is None: continue

    predicted_paths = monte_carlo(   # path predicted through Monte Carlo simulation
        s2o_model, X_val, types_val, index, STEPS_NUMBER, RUNS_NUMBER, WINDOW_SIZE, device, norm_stats, temperature=1
    )

    plot_simulation(      # plot the prediction of a random sample on one axis
        s2o_model,
        X_val,
        types_val,
        index,
        true_path,
        predicted_paths,
        n_steps=STEPS_NUMBER,
        n_runs=RUNS_NUMBER,
        window_size=WINDOW_SIZE,
        norm_stats=norm_stats,
        top_left=TOP_LEFT,
        bottom_right=BOTTOM_RIGHT,
        cell_dims=CELL_DIMENSIONS,
        ax=axes[valid_samples]
    )

    valid_samples += 1    # if last sample was valid, increase the counter

# plot settings
plt.tight_layout()
plt.show()

# **Helpers**

Import needed modules.

In [ ]:
import seaborn as sns

Define helpers.

In [ ]:
def compute_global_stats(dfs_list):
    """
    Helper, computes mean and median for all columns of interest. Used to compute the average time per cell.
    """

    # prepare data
    combined = pl.concat(dfs_list)  # create a single, combined dataframe

    cols_to_analyze = ["x", "y", "SOG", "COG_sin", "COG_cos", "dt", "dSOG", "dCOG"] # keep only columns of interest
    schema = dfs_list[0].collect_schema()
    valid_cols = [c for c in cols_to_analyze if c in schema]

    # define the aggregations
    aggs = []
    for col in valid_cols:
        aggs.append(pl.col(col).mean().alias(f"{col}_mean"))
        aggs.append(pl.col(col).median().alias(f"{col}_median"))

    # execute
    result = combined.select(aggs).collect()

    print("Global dataset statistics:\n")
    print(f"{'Attribute':<10} | {'Mean':<20} | {'Median':<20}")
    print("-" * 56)

    for col in valid_cols:
        mean_val = result[f"{col}_mean"][0]
        med_val = result[f"{col}_median"][0]
        print(f"{col:<10} | {mean_val:<20.2f} | {med_val:<20.2f}")



def analyze_class_distribution(model, val_loader, device="cuda"):
    """
    Allows to compare the distribution of movements in dataset and in the preedictions of the model. Mainly used as a sanity check.
    """

    # prepare structures
    CLASS_NAMES = [
        "0: NW", "1: N", "2: NE",
        "3: W",  "4: Stay", "5: E",
        "6: SW", "7: S",  "8: SE"
    ]

    model.eval()    # notify the model its being evaluated (fundamental to disable dropout)

    pred_counts = torch.zeros(9, dtype=torch.long, device=device)   # to keep counts
    true_counts = torch.zeros(9, dtype=torch.long, device=device)


    # predict
    with torch.no_grad():
        for batch_x, batch_t, batch_y_cls, batch_y_reg in tqdm(val_loader):
            batch_x = batch_x.to(device)    # move to GPU
            batch_t = batch_t.to(device)
            batch_y_cls = batch_y_cls.to(device)

            logits, _ = model(batch_x, batch_t) # get predictions
            preds = torch.argmax(logits, dim=1)

            pred_counts += torch.bincount(preds, minlength=9)   # count
            true_counts += torch.bincount(batch_y_cls, minlength=9)


    # report results
    pred_counts = pred_counts.cpu().numpy()
    true_counts = true_counts.cpu().numpy()
    total_samples = pred_counts.sum()

    df = pd.DataFrame({
        "class": CLASS_NAMES,
        "true_count": true_counts,
        "pred_count": pred_counts,
        "true_%": (true_counts / total_samples) * 100,
        "pred_%": (pred_counts / total_samples) * 100,
        "diff_%": ((pred_counts - true_counts) / total_samples) * 100
    })

    print("\nPrediction distribution statistics:\n")
    print(df.to_string(index=False, float_format="%.2f"))
    print()


    # visualize distribution on a graph
    plt.figure(figsize=(12, 6))

    df_melt = df.melt(id_vars="class", value_vars=["true_%", "pred_%"],
                      var_name="type", value_name="percentage")

    sns.barplot(data=df_melt, x="class", y="percentage", hue="type", palette="viridis")
    plt.title("True Distribution vs Predictions Distribution")
    plt.ylabel("Percentage of Dataset (%)")
    plt.grid(axis='y', alpha=0.3)
    plt.show()

    return df

Compute mean and median of most important features in the dataset.

In [ ]:
compute_global_stats(lazy_dfs);  # obtain statistics

Compute distribution of movements in both dataset and predictions.

In [ ]:
analyze_class_distribution(s2o_model, val_loader, device=device);